In [1]:
import pandas as pd

# 读取数据
df = pd.read_feather("Fund_NAV.feather")
# 确保按日期升序
df['ANN_DATE'] = pd.to_datetime(df['ANN_DATE'], format='%Y%m%d', errors='coerce')
df = df.sort_values('ANN_DATE')

# 更新后的核心字段
fields_to_check = [
    'ANN_DATE',
    'F_NAV_UNIT',
    'F_NAV_DIVACCUMULATED',
    'F_NAV_ADJFACTOR',
    'F_PRT_NETASSET',
    'F_ASSET_MERGEDSHARESORNOT',
    'NETASSET_TOTAL',
    'F_NAV_ADJUSTED',
    'IS_EXDIVIDENDDATE',
    'F_NAV_DISTRIBUTION',
    'S_INFO_ASHARECODE',
    'CUM_NET_ASSET_VALUE'
]

# 创建结果列表
result = []

for field in fields_to_check:
    total_nan_ratio = df[field].isna().mean()  # 整体缺失比例

    # 按年份统计缺失占比
    df['year'] = df['ANN_DATE'].dt.year
    nan_by_year = df.groupby('year')[field].apply(lambda x: x.isna().mean())

    # 判断缺失是否集中在早期（缺失占比 >90%）
    concentrated_years = nan_by_year[nan_by_year > 0.9]
    if len(concentrated_years) > 0:
        concentrated_info = ", ".join(map(str, concentrated_years.index.tolist()))
        # 标记是否可接受
        status = "Acceptable (early missing)"
    else:
        concentrated_info = None
        status = "Needs cleaning"

    result.append({
        'field': field,
        'total_nan_ratio': total_nan_ratio,
        'nan_concentrated_years': concentrated_info,
        'status': status
    })

# 转成 DataFrame 查看
nan_summary = pd.DataFrame(result)
nan_summary = nan_summary.set_index('field')
print(nan_summary)

                           total_nan_ratio  \
field                                        
ANN_DATE                          0.000000   
F_NAV_UNIT                        0.000000   
F_NAV_DIVACCUMULATED              1.000000   
F_NAV_ADJFACTOR                   0.000000   
F_PRT_NETASSET                    0.977380   
F_ASSET_MERGEDSHARESORNOT         0.000000   
NETASSET_TOTAL                    0.985593   
F_NAV_ADJUSTED                    0.000000   
IS_EXDIVIDENDDATE                 0.077946   
F_NAV_DISTRIBUTION                0.077989   
S_INFO_ASHARECODE                 0.000000   
CUM_NET_ASSET_VALUE               0.077990   

                                                      nan_concentrated_years  \
field                                                                          
ANN_DATE                                                                None   
F_NAV_UNIT                                                              None   
F_NAV_DIVACCUMULATED       1999, 20